# Blow ups

In [ ]:
import sys
sys.path.append('../../../src/')

import matplotlib.pyplot as plt

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *

import config
path_save_local = "Img/Local/"

# Energies and values

In [ ]:
A = 0.65
def norm_L2_gauss(sigma, A=A):
    output = A**2 * np.sqrt(np.pi) * sigma
    return np.sqrt(output)

def norm_L2_sech(sigma, A=A):
    output = 2 * A**2 * sigma
    return np.sqrt(output)

def norm_H1_gauss(sigma, A=A):
    output = A**2 * np.sqrt(np.pi) * (1 + 2*sigma**2) / (2 * sigma)
    return np.sqrt(output)

def norm_H1_sech(sigma, A=A):
    output = 2 * A**2 * (1 + 3*sigma**2) / (3 * sigma)
    return np.sqrt(output)


def norm_H2_gauss(sigma, A=A):
    output = A**2 * np.sqrt(np.pi) * (3 + 2*sigma**2 + 4*sigma**4) / (4 * sigma**3)
    return np.sqrt(output)

def norm_H2_sech(sigma, A=A):
    output = 2 * A**2 * (7 + 5*sigma**2 + 15*sigma**4) / (15 * sigma**3)
    return np.sqrt(output)


def E_gauss(sigma, A=A):
    output = A**2 * np.sqrt(np.pi) * (2 - np.sqrt(2) * A**2 + np.sqrt(2) * A**2 * sigma**2) / (8 * sigma)
    return output

def E_sech(sigma, A=A):
    output = A**2 * (5 - 4 * A**2 + 5 * A**2 * sigma**2) / (15 * sigma)
    return output

In [ ]:
# Crea una malla de puntos
a_s = np.linspace(0.1, 1.3, 200)
ss = np.linspace(0.05, 1, 200)
As, Ss = np.meshgrid(a_s, ss)
Energy_gauss = E_gauss(Ss, As)
Energy_sech = E_sech(Ss, As)

# Mapa de calor
plt.imshow(Energy_gauss, extent=[a_s.min(), a_s.max(), ss.min(), ss.max()],
           origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(label="Energy Gaussian")
plt.xlabel(r'$A$')
plt.ylabel(r'$\sigma$')
plt.show()

In [ ]:
# Mapa de calor
plt.imshow(Energy_sech, extent=[a_s.min(), a_s.max(), ss.min(), ss.max()],
           origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(label="Energy Sech")
plt.xlabel(r'$A$')
plt.ylabel(r'$\sigma$')
plt.show()

In [ ]:
sigmas = np.linspace(0.05, 0.5, 1_000)
#A = 0.65
#As = np.linspace(0.62, 0.7, 1_000)
#sigma = 0.1

plt.plot(sigmas, norm_L2_gauss(sigmas), "-", label="Gaussian")
plt.plot(sigmas, norm_L2_sech(sigmas), "-", label="Sech")
#plt.vlines(0.65, 0.065, 0.1, ls="--", color="red", alpha=.5, label="0.65")
plt.ylabel(r"$L^2$-norm")
plt.xlabel(r"$\sigma$")

plt.legend()
plt.show()

plt.plot(sigmas, norm_H1_gauss(sigmas), "-", label="Gaussian")
plt.plot(sigmas, norm_H1_sech(sigmas), "-", label="Sech")
#plt.vlines(0.65, 2.7, 4.5, ls="--", color="red", alpha=.5, label="0.65")
plt.ylabel(r"$H^1$-norm")
plt.xlabel(r"$\sigma$")

plt.legend()
plt.show()

plt.plot(sigmas, norm_H2_gauss(sigmas), "-", label="Gaussian")
plt.plot(sigmas, norm_H2_sech(sigmas), "-", label="Sech")
#plt.vlines(0.65, 350, 670, ls="--", color="red", alpha=.5, label="0.65")
plt.ylabel(r"$H^2$-norm")
plt.xlabel(r"$\sigma$")


plt.legend()
plt.show()

plt.plot(sigmas, E_gauss(sigmas), "-", label="Gaussian")
plt.plot(sigmas, E_sech(sigmas), "-", label="Sech")
#plt.vlines(0.65, 0.8, 1.5, ls="--", color="red", alpha=.5, label="0.65")
plt.ylabel("Energy")
plt.xlabel(r"$\sigma$")


plt.legend()
plt.show()

# Blow up time estimation

In [ ]:
time_file = gzip.GzipFile("../../../scripts/Results/Blow_up/blow_up_time/t_guesses.npy.gz", "r")
blow_up_times = np.load(time_file)

In [ ]:
blow_up_times[blow_up_times!=0][-1]

In [ ]:
plt.plot(blow_up_times[blow_up_times!=0])
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="y")
plt.ylabel(r"$T^*$", rotation='horizontal', ha='right')
plt.xlabel("Iterations")

plt.savefig("Img/Heaven/time_est.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# Gaussian test

## Blow up tester

In [ ]:
A = 0.65 #np.sqrt(2)/2 + 0.01 #1 #0.625 #0.65
sigma = 1/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 15_000
k = 1
Nt = 5_000 #10_000
tf = 2.25e-3
pars = {"k":k}

w_gauss_BL = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_BL.set_init_condition(gaussian)
w_gauss_BL.solve()#method="fixed point")

In [ ]:
w_gauss_BL_max = np_norm(w_gauss_BL.w, np.inf, axis=1)

In [ ]:
sol_lim = np.sqrt(2)/2
cond = w_gauss_BL_max!=0

cond_1_markers = np.where(w_gauss_BL_max<sol_lim)[0]
mark_indices_1 = cond_1_markers[::500]
cond_2_markers = np.where(w_gauss_BL_max>sol_lim)[0]
mark_indices_2 = cond_2_markers

mark_indices = np.append(mark_indices_1, mark_indices_2)

plt.plot(w_gauss_BL.t[cond], w_gauss_BL_max[cond], markevery=mark_indices, markerfacecolor='none', ms=3.5) #

plt.xlabel(r"$t$")
plt.ylabel(r"$||\Psi||_\infty$", rotation='horizontal', ha='right')
plt.hlines(sol_lim, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
#plt.ylim(0.6, 1)
plt.xlim(0, 2.5e-3)
plt.legend(fontsize=6)
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
#plt.savefig(path_save+"Blow_up_As_inf_norm.pdf", dpi=300, bbox_inches = "tight")
plt.show()

## $A$ greater

In [ ]:
A = 2 #np.sqrt(2)/2 + 0.01 #1 #0.625 #0.65
sigma = 1/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 2_000 #10_000
tf = 1.9e-5
pars = {"k":k}

w_gauss_A_sqrt_2 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_A_sqrt_2.set_init_condition(gaussian)
w_gauss_A_sqrt_2.solve()#method="fixed point")

In [ ]:
plt.plot(w_gauss_A_sqrt_2.t, w_gauss_A_sqrt_2.get_energy(), "-")
plt.xlim(0, 2e-6)
plt.ylim(-100, 100)

In [ ]:
virial_A_sqrt_2 = w_gauss_A_sqrt_2.get_virial()

In [ ]:
plt.plot(w_gauss_A_sqrt_2.t, virial_A_sqrt_2, ".-")
plt.xlim(0, 3e-6)
plt.ylim(3.5e-3, 3.6e-3)
plt.show()

In [ ]:
w_gauss_A_sqrt_2.animate_solution("Img/Local/gauss_A_sqrt_2.gif", 100)

In [ ]:
j = 4990
plt.plot(w_gauss_A_sqrt_2.x, norm(w_gauss_A_sqrt_2.w[j]), "-", label=j)
plt.legend()
plt.xlim(-.025, .025)
plt.ylim(0.4, 0.6)
plt.show()

In [ ]:
j = 4700
plt.plot(w_gauss_A_1.x, norm(w_gauss_A_1.w[j]), "-", label=j)
plt.legend()
plt.xlim(-.025, .025)
plt.ylim(0.95, 1.02)
plt.show()

## $\kappa$ tests

In [ ]:
k_1 = 1/8
k_2 = 1
k_3 = 4
k_4 = 8

def A_k(k, eps=1e-1):
    return 1/np.sqrt(2*k) - eps

sigma = 1/10

def gaussian(x, A, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
Nt = 10_000 #10_000
tf = 1.4e-3
pars_1 = {"k":k_1}
pars_2 = {"k":k_2}
pars_3 = {"k":k_3}
pars_4 = {"k":k_4}

A_1 = A_k(k_1, eps=5e-2)
gaussian_k_1_init = lambda x: gaussian(x, A_1)

w_gauss_k_1 = Def.CNFD(L, N, Nt, tf, pars_1, "neumann", order=2)
w_gauss_k_1.set_init_condition(gaussian_k_1_init)
w_gauss_k_1.solve()

In [ ]:
A_2 = A_k(k_2, eps=5e-2)
gaussian_k_2_init = lambda x: gaussian(x, A_2)
tf = 2e-3

w_gauss_k_2 = Def.CNFD(L, N, Nt, tf, pars_2, "neumann", order=2)
w_gauss_k_2.set_init_condition(gaussian_k_2_init)
w_gauss_k_2.solve()#method="fixed point")

In [ ]:
A_3 = A_k(k_3, eps=1e-2)
gaussian_k_3_init = lambda x: gaussian(x, A_3)
tf = 1.3e-3
w_gauss_k_3 = Def.CNFD(L, N, Nt, tf, pars_3, "neumann", order=2)
w_gauss_k_3.set_init_condition(gaussian_k_3_init)
w_gauss_k_3.solve()#method="fixed point")

In [ ]:
A_4 = A_k(k_4, eps=1e-2)
gaussian_k_4_init = lambda x: gaussian(x, A_4)
tf = 1.5e-3
w_gauss_k_4 = Def.CNFD(L, N, Nt, tf, pars_4, "neumann", order=2)
w_gauss_k_4.set_init_condition(gaussian_k_4_init)
w_gauss_k_4.solve()#method="fixed point")

In [ ]:
w_gauss_k_1_max = np_norm(w_gauss_k_1.w, np.inf, axis=1)
w_gauss_k_2_max = np_norm(w_gauss_k_2.w, np.inf, axis=1)
w_gauss_k_3_max = np_norm(w_gauss_k_3.w, np.inf, axis=1)
w_gauss_k_4_max = np_norm(w_gauss_k_4.w, np.inf, axis=1)

In [ ]:
from fractions import Fraction

ks = [k_1, k_2, k_3, k_4]
w_gauss_ks_max = [w_gauss_k_1_max, w_gauss_k_2_max, w_gauss_k_3_max, w_gauss_k_4_max]
w_gauss_ks = [w_gauss_k_1, w_gauss_k_2, w_gauss_k_3, w_gauss_k_4]
ax_mini_insets = [[0.1, 0.25, 0.6, 0.5], [0.15, 0.25, 0.6, 0.5],
                  [0.15, 0.25, 0.6, 0.5], [0.15, 0.25, 0.6, 0.5]]

ax_mini_limits = [[(1e-4, 1.2e-3), (3.5, 5)], [(5e-4, 1.98e-3), (0.3, 1)],
                 [(3e-4, 1.2e-3), (0.1, 0.2)], [(3e-4, 1.4e-3), (0.04, .12)]]

ax_limits = [(-1, 30), (0, 3), (0.05, 0.8), (0.02, .4)]

for j in range(len(ks)):
    k = ks[j]
    w_gauss_k_max = w_gauss_ks_max[j]
    w_gauss_k = w_gauss_ks[j]
    ax_mini_inset = ax_mini_insets[j]
    ax_mini_lims = ax_mini_limits[j]
    ax_limit = ax_limits[j]

    sol_lim = 1/np.sqrt(2*k)+1e-2
    cond = w_gauss_k_max>0

    cond_1_markers = np.where(w_gauss_k_max[cond]<sol_lim)[0]
    mark_indices_1 = cond_1_markers[::300]

    cond_2_markers = np.where(w_gauss_k_max[cond]>sol_lim)[0]
    mark_indices_2 = cond_2_markers[::5]
    mark_indices = np.append(mark_indices_1, mark_indices_2)

    guess = 1/np.sqrt(2*k)**2

    fig = plt.figure()
    ax = fig.add_subplot(111)

    ax.hlines(guess, w_gauss_k.t[w_gauss_k_max>0][0]-10, w_gauss_k.t[w_gauss_k_max>0][-1]+10,
           ls="--", color="gray", label=fr"${Fraction(guess).limit_denominator()}$")

    ax.plot(w_gauss_k.t[w_gauss_k_max>0], w_gauss_k_max[w_gauss_k_max>0]**2, 
         ms=3.5, markevery=mark_indices, label=fr"$\upkappa = -{Fraction(k).limit_denominator()}$", 
            markerfacecolor='none')

    ax_mini = ax.inset_axes(ax_mini_inset)

    ax_mini.plot(w_gauss_k.t[w_gauss_k_max>0], w_gauss_k_max[w_gauss_k_max>0]**2, 
         ms=3.5, markevery=mark_indices, markerfacecolor='none')
    ax_mini.hlines(guess, w_gauss_k.t[w_gauss_k_max>0][0], w_gauss_k.t[w_gauss_k_max>0][-1],
           ls="--", color="gray", label=fr"${Fraction(guess).limit_denominator()}$")

    ax_mini.set_xlim(ax_mini_lims[0])
    ax_mini.set_ylim(ax_mini_lims[1])

    ax_mini.set_xticklabels('')
    ax_mini.set_yticklabels('')
    #ax_mini.grid(linewidth=.5, alpha=.1)
    inset_indicator = ax.indicate_inset_zoom(ax_mini, alpha=.5, label="Zoom", edgecolor="black", lw=.5)

    inset_indicator.connectors[0].set_visible(True)
    inset_indicator.connectors[1].set_visible(True)
    inset_indicator.connectors[2].set_visible(True)
    inset_indicator.connectors[3].set_visible(True)
    
    ax.set_ylim(ax_limit)
    ax.set_xlim(w_gauss_k.t[w_gauss_k_max>0][0], w_gauss_k.t[w_gauss_k_max>0][-1])
    ax.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"$||\Psi||^2_\infty$", rotation='horizontal', ha='right')
    #ax.grid(linewidth=.5, alpha=.1)

    plt.legend()
    plt.savefig(path_save_local+f"Gauss_k/Blow_up_k_{k}.pdf", dpi=300, bbox_inches = "tight")
    plt.show()

## Gaussian $\sigma$'s comparisson

In [ ]:
A = 0.631 #0.65 #0.625 #0.65
sigma_1 = 1/10
sigma_2 = 2 * sigma_1
sigma_3 = sigma_1 / 2

def gaussian(x, sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

gauss_1 = lambda x: gaussian(x, sigma=sigma_1)
gauss_2 = lambda x: gaussian(x, sigma=sigma_2)
gauss_3 = lambda x: gaussian(x, sigma=sigma_3)

In [ ]:
L = np.pi
N = 10_000
k = 1 # 5/3
Nt = 10_000 #10_000
tf = 1e-2 #2.25e-3
pars = {"k":k}

w_gauss_1 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_1.set_init_condition(gauss_1)
w_gauss_1.solve()#method="fixed point")

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 10_000 #10_000
tf = 1e-3 #2.25e-3
pars = {"k":k}

w_gauss_2 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_2.set_init_condition(gauss_2)
w_gauss_2.solve()

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 2_000 #10_000
tf = 3e-2 #2.25e-3
pars = {"k":k}

w_gauss_3 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_3.set_init_condition(gauss_3)
w_gauss_3.solve()

In [ ]:
w_gauss_3.animate_solution("test.gif", 50)

In [ ]:
w_gauss_3_max = np_norm(w_gauss_3.w, np.inf, axis=1)

In [ ]:
plt.plot(w_gauss_3.t, w_gauss_3_max**2, markevery=20, ms=2)
#plt.ylim(0.35, 1)
plt.show()

In [ ]:
A = 0.64 #0.65 #0.625 #0.65 #0.638
sigma = 0.5

def gaussian(x, sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

gauss = lambda x: gaussian(x, sigma=sigma)

L = np.pi
N = 10_000
k = 1
Nt = 2_000 #10_000
tf = 1e-1 #2.25e-3
pars = {"k":k}

w_gauss = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss.set_init_condition(gauss)
w_gauss.solve()

In [ ]:
w_gauss_max = np_norm(w_gauss.w, np.inf, axis=1)

In [ ]:
plt.plot(w_gauss.t, w_gauss_max**2, markevery=20, ms=2)
plt.ylim(0.35, 1)
plt.show()

## Gif creator

In [ ]:
# Create figure
N_frames = 150 # CAMBIAR A 10

x_indices = np.abs(w_gauss_1.x)<1

fig, ax = plt.subplots()
line1, = ax.plot(w_gauss_1.x[x_indices], norm(w_gauss_1.w[0][x_indices]), "-", label=r"$\sigma = 0.1$")
line2, = ax.plot(w_gauss_2.x[x_indices], norm(w_gauss_2.w[0][x_indices]), "-", label=r"$\sigma = 0.2$")
line3, = ax.plot(w_gauss_3.x[x_indices], norm(w_gauss_3.w[0][x_indices]), "-", label=r"$\sigma = 0.05$")

ax.set_title(r"Time $t = 0$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$|\psi|^2$")
ax.set_xlim(-1, 1)
ax.set_ylim(-0.1, 0.7)

fig.tight_layout()

# Animate function
def animate(j):
    dt = w_gauss_1.dt
    t_plot = w_gauss_1.t[j]

    x_indices = np.abs(w_gauss_1.x)<1

    # Update plot
    line1.set_data(w_gauss_1.x[x_indices], norm(w_gauss_1.w[j][x_indices]))
    line2.set_data(w_gauss_2.x[x_indices], norm(w_gauss_2.w[j][x_indices]))
    line3.set_data(w_gauss_3.x[x_indices], norm(w_gauss_3.w[j][x_indices]))
    ax.set_title(r"Time $t =$ {}".format(sci_notation(t_plot, 3)))

    ax.legend()

    return line1, line2, line3,

# Animación
ani = animation.FuncAnimation(fig, animate, frames=np.arange(0, Nt, N_frames), interval=150)

ani.save("Img/Local/gauss_sigmas.gif")
plt.show()

### Guassian $\sigma = 0.2$ and smaller $A$

In [ ]:
A = 0.63
sigma_1 = 1/10
sigma_2 = 2 * sigma_1

def gaussian(x, sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

gauss_1_A = lambda x: gaussian(x, sigma=sigma_1)
gauss_2_A = lambda x: gaussian(x, sigma=sigma_2)

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 4_000 #10_000
tf = 1.8e-2 #2.25e-3
pars = {"k":k}

w_gauss_1_A = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_1_A.set_init_condition(gauss_1_A)
w_gauss_1_A.solve()

In [ ]:
w_gauss_2_A = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss_2_A.set_init_condition(gauss_2_A)
w_gauss_2_A.solve()

In [ ]:
# Create figure
N_frames = 150

x_indices = np.abs(w_gauss_1_A.x)<1

fig, ax = plt.subplots()
line1, = ax.plot(w_gauss_1_A.x[x_indices], norm(w_gauss_1_A.w[0][x_indices]), "-", label=r"$\sigma = 0.1$")
line2, = ax.plot(w_gauss_2_A.x[x_indices], norm(w_gauss_2_A.w[0][x_indices]), "-", label=r"$\sigma = 0.2$")

ax.set_title(r"Time $t = 0$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$|\psi|^2$")
ax.set_xlim(-1, 1)
ax.set_ylim(-0.1, 0.7)

fig.tight_layout()

# Animate function
def animate(j):
    dt = w_gauss_1.dt
    t_plot = w_gauss_1_A.t[j]

    x_indices = np.abs(w_gauss_1_A.x)<1

    # Update plot
    line1.set_data(w_gauss_1_A.x[x_indices], norm(w_gauss_1_A.w[j][x_indices]))
    line2.set_data(w_gauss_2_A.x[x_indices], norm(w_gauss_2_A.w[j][x_indices]))
    ax.set_title(r"Time $t =$ {}".format(sci_notation(t_plot, 3)))

    ax.legend()

    return line1, line2,

# Animación
ani = animation.FuncAnimation(fig, animate, frames=np.arange(0, Nt, N_frames), interval=150)

ani.save("Img/Local/gauss_sigmas_A_0_63.gif")
plt.show()

# Guassian-cosine tests

In [ ]:
A = 0.65 #0.65
sigma = 0.1 #1/10
l = 15

def gauss_cos_init(x):
    output = A * np.exp(-x**2/(2 * sigma**2)) * np.cos(l*x)
    return output

In [ ]:
x_plot = np.linspace(-np.pi, np.pi, 1000)

plt.plot(x_plot, norm(gauss_cos_init(x_plot)), "-", label="cos")
plt.plot(x_plot, norm(gaussian(x_plot)), "-", label="gauss")
plt.plot(x_plot, norm(sech_init(x_plot)), "-", label="sech")
plt.legend()
plt.xlim(-0.5, 0.5)
plt.show()

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 5_000 #10_000
tf = 1.2e-3
pars = {"k":k}

w_cos = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_cos.set_init_condition(gauss_cos_init)
w_cos.solve()#method="fixed point")

In [ ]:
w_cos.animate_solution("Img/Local/gauss_cos_l_15.gif", 50)

# Hyperbolic secant tests

In [ ]:
A = 0.65 #0.65
sigma = 0.1 #1/10

def sech_init(x):
    output = A * sech(x/(sigma))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 5_000 #10_000
tf = 1e-2 #2.25e-3
pars = {"k":k}

w_sech = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_sech.set_init_condition(sech_init)
w_sech.solve()#method="fixed point")

In [ ]:
w_sech.animate_solution("Img/Local/sech.gif", 50)

# Generate solutions

In [ ]:
A = 0.625 #np.sqrt(2)/2 + 0.01 #1 #0.625 #0.65
sigma = 1/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 5_000
k = 1
Nt = 5_000 #10_000
tf = 1e-1 #2.25e-3
pars = {"k":k}

w_gauss = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_gauss.set_init_condition(gaussian)
w_gauss.solve()#method="fixed point")

In [ ]:
w_gauss.animate_solution("test.gif", 50)

In [ ]:
w_gauss_max = np_norm(w_gauss.w, np.inf, axis=1)

In [ ]:
plt.plot(w_gauss.t, w_gauss_max)
plt.ylim(0.55, 0.8)
plt.xlim(0, 3.5e-3)
plt.show()

In [ ]:
N_2 = 5_000

w_gauss_N_2 = Def.CNFD(L, N_2, Nt, tf, pars, "neumann", order=2)
w_gauss_N_2.set_init_condition(gaussian)
w_gauss_N_2.solve()

# Load solutions (Deprecated)

In [ ]:
w_gauss_N_1_Nt_1_file = gzip.GzipFile("Gauss_N_10000_Nt_10000/w_gauss_N_10000.npy.gz", "r")
energy_N_1_Nt_1_file = gzip.GzipFile("Gauss_N_10000_Nt_10000/energy_N_10000.npy.gz", "r")
mass_N_1_Nt_1_file = gzip.GzipFile("Gauss_N_10000_Nt_10000/mass_N_10000.npy.gz", "r")
w_gauss_max_N_1_Nt_1_file = gzip.GzipFile("Gauss_N_10000_Nt_10000/w_gauss_max_N_10000.npy.gz", "r")


w_gauss_N_1_Nt_1 = np.load(w_gauss_N_1_Nt_1_file, allow_pickle=True)
energy_N_1_Nt_1 = np.load(energy_N_1_Nt_1_file)
mass_N_1_Nt_1 = np.load(mass_N_1_Nt_1_file)
w_gauss_max_N_1_Nt_1 = np.load(w_gauss_max_N_1_Nt_1_file)

In [ ]:
w_gauss_N_1_Nt_2_file = gzip.GzipFile("Gauss_N_10000_Nt_5000/w_gauss_N_10000.npy.gz", "r")
energy_N_1_Nt_2_file = gzip.GzipFile("Gauss_N_10000_Nt_5000/energy_N_10000.npy.gz", "r")
mass_N_1_Nt_2_file = gzip.GzipFile("Gauss_N_10000_Nt_5000/mass_N_10000.npy.gz", "r")
w_gauss_max_N_1_Nt_2_file = gzip.GzipFile("Gauss_N_10000_Nt_5000/w_gauss_max_N_10000.npy.gz", "r")


w_gauss_N_1_Nt_2 = np.load(w_gauss_N_1_Nt_2_file, allow_pickle=True)
energy_N_1_Nt_2 = np.load(energy_N_1_Nt_2_file)
mass_N_1_Nt_2 = np.load(mass_N_1_Nt_2_file)
w_gauss_max_N_1_Nt_2 = np.load(w_gauss_max_N_1_Nt_2_file)

In [ ]:
w_gauss_N_2_Nt_1_file = gzip.GzipFile("Gauss_N_5000_Nt_10000/w_gauss_N_5000.npy.gz", "r")
energy_N_2_Nt_1_file = gzip.GzipFile("Gauss_N_5000_Nt_10000/energy_N_5000.npy.gz", "r")
mass_N_2_Nt_1_file = gzip.GzipFile("Gauss_N_5000_Nt_10000/mass_N_5000.npy.gz", "r")
w_gauss_max_N_2_Nt_1_file = gzip.GzipFile("Gauss_N_5000_Nt_10000/w_gauss_max_N_5000.npy.gz", "r")


w_gauss_N_2_Nt_1 = np.load(w_gauss_N_2_Nt_1_file, allow_pickle=True)
energy_N_2_Nt_1 = np.load(energy_N_2_Nt_1_file)
mass_N_2_Nt_1 = np.load(mass_N_2_Nt_1_file)
w_gauss_max_N_2_Nt_1 = np.load(w_gauss_max_N_2_Nt_1_file)

In [ ]:
w_gauss_N_2_Nt_2_file = gzip.GzipFile("Gauss_N_5000_Nt_5000/w_gauss_N_5000.npy.gz", "r")
energy_N_2_Nt_2_file = gzip.GzipFile("Gauss_N_5000_Nt_5000/energy_N_5000.npy.gz", "r")
mass_N_2_Nt_2_file = gzip.GzipFile("Gauss_N_5000_Nt_5000/mass_N_5000.npy.gz", "r")
w_gauss_max_N_2_Nt_2_file = gzip.GzipFile("Gauss_N_5000_Nt_5000/w_gauss_max_N_5000.npy.gz", "r")


w_gauss_N_2_Nt_2 = np.load(w_gauss_N_2_Nt_2_file, allow_pickle=True)
energy_N_2_Nt_2 = np.load(energy_N_2_Nt_2_file)
mass_N_2_Nt_2 = np.load(mass_N_2_Nt_2_file)
w_gauss_max_N_2_Nt_2 = np.load(w_gauss_max_N_2_Nt_2_file)

In [ ]:
Nt_1 = 10_000
Nt_2 = 5_000
tf = 2.25e-3
t_1 = np.linspace(0, tf, Nt_1)
t_2 = np.linspace(0, tf, Nt_2)

In [ ]:
plt.plot(t_1, w_gauss_max_N_1_Nt_1, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=10000$")
plt.plot(t_1, w_gauss_max_N_2_Nt_1, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=5000$")

plt.hlines(np.sqrt(2)/2, 0, t[-1], color="gray", ls="--")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.xlim(2e-3, 2.25e-3)
plt.legend()
#plt.ylim(0.65, 1)
#plt.savefig("gaussian.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
plt.plot(t_1, w_gauss_max_N_1_Nt_1, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$Nt=10000$")
plt.plot(t_2[w_gauss_max_N_1_Nt_2>0][:-1], w_gauss_max_N_1_Nt_2[w_gauss_max_N_1_Nt_2>0][:-1], "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$Nt=5000$")

plt.hlines(np.sqrt(2)/2, 0, t[-1], color="gray", ls="--")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.xlim(2e-3, 2.25e-3)
plt.legend()
#plt.ylim(0.65, 1)
#plt.savefig("gaussian.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
plt.plot(t_1, w_gauss_max_N_2_Nt_1, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$Nt=10000$")
plt.plot(t_2, w_gauss_max_N_2_Nt_2, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$Nt=5000$")

plt.hlines(np.sqrt(2)/2, 0, t[-1], color="gray", ls="--")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.xlim(2e-3, 2.25e-3)
plt.legend()
#plt.ylim(0.65, 1)
#plt.savefig("gaussian.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
plt.plot(t, energy_N_1, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=10000$")
plt.plot(t, energy_N_2, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=5000$")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.xlim(2e-3, 2.25e-3)
plt.legend()
#plt.ylim(0.65, 1)
#plt.savefig("gaussian.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
plt.plot(t, mass_N_1, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=10000$")
plt.plot(t, mass_N_2, "x-", lw=.5, markeredgewidth=.5, ms=3, label=r"$N=5000$")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.legend()
#plt.savefig("gaussian.pdf", dpi=300, bbox_inches = "tight")
plt.show()